In [1]:
import pandas as pd
from pathlib import Path

BASE = Path("../data/raw")

# better table display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 100)

# warning fix (mixed types)
import warnings
warnings.filterwarnings("ignore")

In [2]:
df_raw = pd.read_csv(BASE / "core/AVONET_Raw_Data.csv", encoding="latin1")
df_birdLife = pd.read_csv(BASE / "core/AVONET1_BirdLife.csv", encoding="latin1")
df_ebird = pd.read_excel(BASE / "core/AVONET2_eBird.xlsx", sheet_name=1 , header=0)
df_birdtree = pd.read_excel( BASE / "core/AVONET3_BirdTree.xlsx", sheet_name=1 , header=0 )
df_cross_BL_BT = pd.read_csv( BASE / "metadata/BirdLife_BirdTree_Taxonomy_Relation.csv", encoding="latin1")
# df_family_specimen = pd.read_csv( BASE / "metadata/Family_Specimen_Stats.csv",encoding="latin1")
# df_source_details = pd.read_csv( BASE / "metadata/Source_Details.csv", encoding="latin1" )
# df_history = pd.read_csv(BASE / "metadata/Trait_Database_History.csv", encoding="latin1")

# <span style="font-weight:900  ">Functional Dependancy</span>  

###  Avibase.ID1 --> Species1  <span style="font-weight:300 ; color:#ff0000">(of  birdLife)</span> 

In [3]:
violations = ( df_birdLife.groupby("Avibase.ID1")["Species1"].nunique().gt(1) )
bad_ids = violations[violations].index
print("Number of violating Avibase IDs:", len(bad_ids))

Number of violating Avibase IDs: 0


### Species1 --> Avibase.ID1 <span style="font-weight:300 ; color:#ff0000">(of  birdLife)</span> 

In [4]:

violations = ( df_birdLife.groupby("Species1")["Avibase.ID1"].nunique().gt(1) )
bad_ids = violations[violations].index

print("Number of violating Avibase IDs:", len(bad_ids))

Number of violating Avibase IDs: 0


### Avibase.ID2 --> Species2     <span style="font-weight:300 ; color:#ff0000"> ( of ebird )</span>   

In [5]:

violations = ( df_ebird.groupby("Avibase.ID2")["Species2"].nunique().gt(1) )
bad_ids = violations[violations].index

print("Number of violating Avibase IDs:", len(bad_ids))

Number of violating Avibase IDs: 0


### Species2 -> Avibase.ID2    <span style="font-weight:300 ; color:#ff0000"> ( of ebird )</span>

In [6]:
violations = ( df_ebird.groupby("Species2")["Avibase.ID2"].nunique().gt(1) )
bad_ids = violations[violations].index

print("Number of violating Avibase IDs:", len(bad_ids))

Number of violating Avibase IDs: 0


#  Conclusion 
- birdLife : Species1 <--> Avibase.ID1 is bi-directional functional dependency
- ebird    : Species2 <--> Avibase.ID2 is bi-directional functional dependency

# <span style="font-weight:900 "> Missing AvibaseIDs </span>

- AvibaseIDs which are present in raw_Data but not in birdLife

In [7]:
birdlife_ids = ( df_birdLife["Avibase.ID1"].astype(str).str.strip().str.upper() )
raw_ids = ( df_raw["Avibase.ID"].astype(str).str.strip().str.upper() )

invalid_tokens = {"", "N", "NA", "NAN", "NONE", "NULL"}

raw_ids_clean = raw_ids[~raw_ids.isin(invalid_tokens)]
missing_in_birdlife = raw_ids_clean[~raw_ids_clean.isin(birdlife_ids)]

print("Raw_Data IDs not in BirdLife:", missing_in_birdlife.nunique())

Raw_Data IDs not in BirdLife: 576


- AvibaseIDs which are present in raw_Data but not in eBird

In [8]:

ebird_ids = ( df_ebird["Avibase.ID2"].astype(str).str.strip().str.upper() )
raw_ids = (df_raw["Avibase.ID"].astype(str).str.strip().str.upper() )

invalid_tokens = {"", "N", "NA", "NAN", "NONE", "NULL"}
raw_ids_clean = raw_ids[~raw_ids.isin(invalid_tokens)]
missing_in_ebird = raw_ids_clean[~raw_ids_clean.isin(ebird_ids)]

print("Raw_Data IDs not in BirdLife:", missing_in_ebird.nunique())

Raw_Data IDs not in BirdLife: 1136


- AvibaseIDs which are present in raw_Data but not in union of BirdLife and eBird

In [9]:
def clean_ids(series):
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .replace({"": None, "N": None, "NA": None, "NAN": None, "NONE": None, "NULL": None})
        .dropna()
    )

birdlife_ids = set(clean_ids(df_birdLife["Avibase.ID1"]))
ebird_ids = set(clean_ids(df_ebird["Avibase.ID2"]))
raw_ids = clean_ids(df_raw["Avibase.ID"])

reference_ids = birdlife_ids | ebird_ids
missing_in_both = raw_ids[~raw_ids.isin(reference_ids)]

print("Raw IDs missing in BOTH BirdLife and eBird:", missing_in_both.nunique())

Raw IDs missing in BOTH BirdLife and eBird: 82


In [17]:
print( "Null values in BirdLife Total.individuals:" , df_birdLife["Total.individuals"].isnull().sum() ,"\n")

print( "Sum of Total.individuals in BirdLife:" , df_birdLife['Total.individuals'].sum() )

Null values in BirdLife Total.individuals: 0 

Sum of Total.individuals in BirdLife: 89652


# Conclusion
- BirdLife is the authenticated, species-level aggregate of the raw specimen data.
- Reason: The total individual count aligns perfectly with the raw records and the functional dependency between identifiers is rock-solid, making this our most reliable "Source of Truth" for the analysis

---
- raw_data may help use to define the map between all taxonomies and avibaseId